# RALF on Kaggle / Colab

Run **RALF** (Retrieval-Augmented Layout Transformer, CVPR 2024) on a cloud GPU.

This notebook:
1. clones your RALF fork,
2. wires the 13 GB `cache/` you uploaded as a Kaggle Dataset (or Google Drive on Colab),
3. installs the runtime deps (leaves torch alone -- Kaggle/Colab already ship a CUDA torch 2.x),
4. verifies the pretrained checkpoint (`ralf_refinement_cgl/gen_final_model.pt`) loads.

**Prereq:** replace `REPO_URL` in cell 2 with your fork, and upload `cache/` as a Kaggle Dataset (cell 3).

In [ ]:
# 1) Clone (or update) the repo.  <-- CHANGE THIS to your fork
REPO_URL = "https://github.com/<your-user>/RALF.git"
REPO_DIR = "/kaggle/working/RALF"  # Colab: use /content/RALF
import os, subprocess
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(subprocess.run(["git", "-C", REPO_DIR, "pull"], text=True).stdout)
else:
    print(subprocess.run(["git", "clone", REPO_URL, REPO_DIR], text=True).stdout)
print("repo at:", REPO_DIR)

## 2) Provide the `cache/` directory (13 GB)

**Kaggle:** add this notebook's **Input** -> *Your Datasets* -> upload the `cache/` directory
(or a zip of it). It will appear at `/kaggle/input/<dataset-slug>/cache`. The bootstrap script
auto-detects it and symlinks `cache/dataset`, `cache/PRECOMPUTED_WEIGHT_DIR`, `cache/eval_gt_features`,
`cache/training_logs`. `training_logs` is ~5.8 GB; drop it from the dataset if you only need inference.

**Colab:** put `cache/` on Drive at `/content/drive/MyDrive/RALF/cache` and run
`from google.colab import drive; drive.mount('/content/drive')` first.

In [ ]:
# 3) Bootstrap env.  Set INSTALL=True the first time (installs deps, keeps torch).
INSTALL = True
os.chdir(REPO_DIR)
cmd = "source scripts/kaggle/bootstrap.sh" + (" --install" if INSTALL else "")
print(subprocess.run(["bash", "-lc", cmd], text=True, capture_output=True).stdout)

In [ ]:
# 4) Verify environment + data presence
import os, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
for p in ["cache/dataset/cgl", "cache/PRECOMPUTED_WEIGHT_DIR/fidnet/cgl",
          "cache/training_logs/ralf_refinement_cgl/gen_final_model.pt"]:
    print(f"{'OK ' if os.path.exists(p) else 'MISSING'} {p}")

In [ ]:
# 5) Load the pretrained checkpoint we inspected and summarise its modules
from collections import Counter
ckpt = "cache/training_logs/ralf_refinement_cgl/gen_final_model.pt"
sd = torch.load(ckpt, map_location="cpu", weights_only=False)
print(type(sd).__name__, "|", len(sd), "tensors")
for root, n in Counter(k.split(".")[0] for k in sd).most_common():
    print(f"  {root:22s} {n}")
print("flag_img:", sd.get("flag_img"), "| flag_user_const:", sd.get("flag_user_const"))

In [ ]:
# 6) (optional) Reproduce inference + evaluation for the refinement RALF checkpoint
cmd = "source scripts/kaggle/bootstrap.sh && bash scripts/run_job/eval_inference.sh 0 ./cache/training_logs/ralf_refinement_cgl refinement cgl"
p = subprocess.run(["bash", "-lc", cmd], text=True)
print("returncode:", p.returncode)